In [1]:
# --- Step 1: Import Libraries ---
import pandas as pd
import numpy as np
from google.colab import files

# --- Step 2: Upload the Dataset ---
uploaded = files.upload()  # This will prompt you to upload the file

# Get the uploaded file name
filename = list(uploaded.keys())[0]
print(f"Uploaded file: {filename}")

# Load the dataset
df = pd.read_csv(filename)
print("Dataset Loaded Successfully!")
print(df.head())

Saving Loan.txt to Loan (1).txt
Uploaded file: Loan (1).txt
Dataset Loaded Successfully!
  loanId\tmemberId\tdate\tpurpose\tisJointApplication\tloanAmount\tterm\tinterestRate\tmonthlyPayment\tgrade\tloanStatus
0  1888978\t2305095\t12/10/2014\tdebtconsolidatio...                                                                    
1  1299695\t2610493\t9/15/2014\tdebtconsolidation...                                                                    
2  1875016\t2491679\t9/11/2014\tdebtconsolidation...                                                                    
3  1440478\t2092798\t4/22/2016\thomeimprovement\t...                                                                    
4  1124634\t2633077\t2/3/2016\tdebtconsolidation\...                                                                    


In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, confusion_matrix
from tabulate import tabulate
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.svm import LinearSVC
import warnings

# ------------------- 0. Suppress warnings -------------------
warnings.filterwarnings("ignore")

# ------------------- 1. Common preprocessing -------------------
df = pd.read_csv("Loan.txt", sep="\t")

# Target
df["Default"] = df["loanStatus"].apply(lambda x: 1 if x in ["Default", "Charged Off", "Late"] else 0)

# Features & target
X = df.drop(columns=["loanId", "memberId", "date", "loanStatus", "Default"])
y = df["Default"]

# One-hot encode categorical
X = pd.get_dummies(X, drop_first=True)

# Impute missing values
X = SimpleImputer(strategy="median").fit_transform(X)

# Train/Test split
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

# Scale features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

# ------------------- 2. Function to evaluate model -------------------
def evaluate_model_train_test(model, model_name):
    model.fit(X_train, y_train)

    # Train predictions
    y_train_pred = model.predict(X_train)
    train_accuracy = accuracy_score(y_train, y_train_pred)

    # Test predictions
    y_test_pred = model.predict(X_test)
    tn, fp, fn, tp = confusion_matrix(y_test, y_test_pred).ravel()
    test_accuracy = accuracy_score(y_test, y_test_pred)

    results = {
        "Train Accuracy": train_accuracy,
        "Test Accuracy": test_accuracy,
        "Sensitivity (Recall)": recall_score(y_test, y_test_pred),
        "Specificity": tn / (tn + fp),
        "Precision": precision_score(y_test, y_test_pred),
        "F1-Score": f1_score(y_test, y_test_pred)
    }

    table = [[
        model_name,
        round(results["Train Accuracy"], 4),
        round(results["Test Accuracy"], 4),
        round(results["Sensitivity (Recall)"], 4),
        round(results["Specificity"], 4),
        round(results["Precision"], 4),
        round(results["F1-Score"], 4)
    ]]
    headers = ["Model", "Train Acc", "Test Acc", "Sensitivity", "Specificity", "Precision", "F1-Score"]
    print(tabulate(table, headers=headers, tablefmt="grid"))

# ------------------- 3. Models -------------------

# Logistic Regression
evaluate_model_train_test(LogisticRegression(max_iter=1000, random_state=42), "Logistic Regression")

# KNN
evaluate_model_train_test(KNeighborsClassifier(n_neighbors=5), "KNN")

# Decision Tree
evaluate_model_train_test(DecisionTreeClassifier(random_state=42), "Decision Tree")

# Random Forest
evaluate_model_train_test(RandomForestClassifier(n_estimators=100, random_state=42), "Random Forest")

# Gradient Boosting
evaluate_model_train_test(GradientBoostingClassifier(n_estimators=100, random_state=42), "Gradient Boosting")

# XGBoost
evaluate_model_train_test(XGBClassifier(eval_metric='logloss', random_state=42), "XGBoost")

# Linear SVM
evaluate_model_train_test(LinearSVC(max_iter=5000, random_state=42), "Linear SVM")


+---------------------+-------------+------------+---------------+---------------+-------------+------------+
| Model               |   Train Acc |   Test Acc |   Sensitivity |   Specificity |   Precision |   F1-Score |
+=====================+=============+============+===============+===============+=============+============+
| Logistic Regression |      0.9112 |     0.9125 |        0.2153 |        0.9899 |      0.7037 |     0.3298 |
+---------------------+-------------+------------+---------------+---------------+-------------+------------+
+---------+-------------+------------+---------------+---------------+-------------+------------+
| Model   |   Train Acc |   Test Acc |   Sensitivity |   Specificity |   Precision |   F1-Score |
+=========+=============+============+===============+===============+=============+============+
| KNN     |      0.9203 |      0.904 |        0.2207 |        0.9799 |      0.5498 |     0.3149 |
+---------+-------------+------------+---------------+----